In [ ]:
import pandas as pd
import sqlite3
import glob
import os

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
SIMULATIONS = {
    "custom_logistics_sim": os.path.join(BASE_DIR, "custom_logistics_sim"),
    "custom_logistics_sim_wi": os.path.join(BASE_DIR, "custom_logistics_sim_wi"),
    "custom_logistics_sim_wi_3": os.path.join(BASE_DIR, "custom_logistics_sim_wi_3"),
}

In [3]:
for sim_name, sim_dir in SIMULATIONS.items():
    sql_path = os.path.join(BASE_DIR, f"{sim_name}.sqlite")

    # Remove existing sqlite file if present
    if os.path.exists(sql_path):
        os.remove(sql_path)

    # Find event and object type CSVs
    eventTypeTableFilenames = [
        fn for fn in glob.glob(os.path.join(sim_dir, "event_*.csv"))
        if not fn.endswith("event_object.csv")
    ]
    objectTypeTableFilenames = [
        fn for fn in glob.glob(os.path.join(sim_dir, "object_*.csv"))
        if not fn.endswith("object_object.csv")
    ]

    # Load all tables
    TABLES = dict()
    TABLES["event"] = pd.read_csv(os.path.join(sim_dir, "event.csv"), sep=";")
    TABLES["event_object"] = pd.read_csv(os.path.join(sim_dir, "event_object.csv"), sep=";")
    TABLES["object"] = pd.read_csv(os.path.join(sim_dir, "object.csv"), sep=";")
    TABLES["object_object"] = pd.read_csv(os.path.join(sim_dir, "object_object.csv"), sep=";")
    for fn in eventTypeTableFilenames:
        table_name = os.path.basename(fn).split(".")[0]
        TABLES[table_name] = pd.read_csv(fn, sep=";")
    for fn in objectTypeTableFilenames:
        table_name = os.path.basename(fn).split(".")[0]
        TABLES[table_name] = pd.read_csv(fn, sep=";")

    # Write to sqlite
    conn = sqlite3.connect(sql_path)
    for tn, df in TABLES.items():
        df.to_sql(tn, conn, index=False)
    conn.close()

    print(f"Created {sql_path} with {len(TABLES)} tables")

Created /Users/doclam/Documents/code/federated-sfd/data/whatIF/custom_logistics_sim.sqlite with 27 tables
Created /Users/doclam/Documents/code/federated-sfd/data/whatIF/custom_logistics_sim_wi.sqlite with 27 tables
